In [26]:
# 1. 먼저 pyreadstat 패키지 설치
!pip install pyreadstat

# 2. 구글 드라이브 연동
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import pyreadstat

# 3. 경로 설정
DATA_DIR = '/content/drive/MyDrive/hdac_data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import glob

# 구글 드라이브 전체에서 파일 위치 찾기
found_files = glob.glob('/content/drive/**/*.sas7bdat', recursive=True)
print("🔍 파일이 실제로 존재하는 경로:", found_files)

🔍 파일이 실제로 존재하는 경로: ['/content/drive/MyDrive/hdac_data/hn24_all.sas7bdat']


In [28]:
DATA_DIR = '/content/drive/MyDrive/hdac_data'

target_cols = ['age', 'sex', 'kstrata', 'psu', 'wt_ntr']
df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols)

print("전체 데이터 크기 (행, 열):", df.shape)

전체 데이터 크기 (행, 열): (6997, 5)


In [29]:
import pandas as pd
import pyreadstat

# 1. 파일 경로 설정
DATA_DIR = '/content/drive/MyDrive/hdac_data'

# 2. 파일 내에 실제로 존재하는 모든 열 이름(대소문자 포함)을 먼저 읽어와 확인합니다.
_, meta_all = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', metadataonly=True)
available_cols = meta_all.column_names

# 3. 우리가 불러오고자 하는 타겟 변수 목록
target_cols = [
    'age', 'sex', 'kstrata', 'psu', 'wt_ntr',
    'genertn',    # 1인 가구 여부
    'HE_khei',    # 식생활평가지수
    'incm',       # 소득
    'edu',        # 교육수준
    'BD1_11',     # 음주
    'pa_aerobic'  # 유산소 운동
]

# 실제 파일에 존재하는 변수들만 골라냅니다 (대소문자 구문 문제 대응)
valid_target_cols = []
missing_cols = []

for col in target_cols:
    # 대소문자 구분 없이 매칭되는 변수가 있는지 확인
    matched = [c for c in available_cols if c.lower() == col.lower()]
    if matched:
        valid_target_cols.append(matched[0])
    else:
        missing_cols.append(col)

print("✅ 불러올 유효한 변수들:", valid_target_cols)
if missing_cols:
    print("⚠️ 데이터에 없는 변수들:", missing_cols)

# 4. 유효한 변수들만 골라서 데이터 로드
df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=valid_target_cols)
print("확장된 데이터 크기 (행, 열):", df.shape)

# 5. 대상 선정 단계별 인원 확인
print("=" * 40)
print(" [대상 선정 단계별 인원 확인]")
print("=" * 40)
print(f"1단계 - 초기 전체 데이터 수: {len(df):,}명")

if 'wt_ntr' in df.columns:
    df_filtered = df.dropna(subset=['wt_ntr'])
    print(f"2단계 - 영양조사 가중치(wt_ntr) 결측 제외 후: {len(df_filtered):,}명")
else:
    df_filtered = df
    print("2단계 - 가중치 변수 없음 (전체 유지)")

# 6. 주요 변수 원 코드 빈도표 확인
print("\n" + "=" * 40)
print(" [주요 변수 원 코드 및 빈도 검증]")
print("=" * 40)

if 'genertn' in df_filtered.columns:
    print("--- 1인 가구 여부(genertn) 원 코드 빈도표 ---")
    print(df_filtered['genertn'].value_counts(dropna=False))

if 'HE_khei' in df_filtered.columns:
    print("\n--- 식생활평가지수(HE_khei) 요약 통계량 ---")
    print(df_filtered['HE_khei'].describe())

✅ 불러올 유효한 변수들: ['age', 'sex', 'kstrata', 'psu', 'wt_ntr', 'genertn', 'incm', 'edu', 'BD1_11', 'pa_aerobic']
⚠️ 데이터에 없는 변수들: ['HE_khei']
확장된 데이터 크기 (행, 열): (6997, 10)
 [대상 선정 단계별 인원 확인]
1단계 - 초기 전체 데이터 수: 6,997명
2단계 - 영양조사 가중치(wt_ntr) 결측 제외 후: 6,805명

 [주요 변수 원 코드 및 빈도 검증]
--- 1인 가구 여부(genertn) 원 코드 빈도표 ---
genertn
4.0    2807
2.0    1622
1.0    1056
5.0     625
7.0     323
6.0     278
3.0      85
9.0       9
Name: count, dtype: int64


In [30]:
# 이미 로드된 통합 데이터프레임(df)의 열 이름 중에서 'khei'가 포함된 변수 찾기
khei_candidates = [col for col in df.columns if 'khei' in col.lower()]
print("🔍 'khei'가 포함된 실제 변수 후보:", khei_candidates)

if khei_candidates:
    # 찾은 변수를 분석용 표준 이름인 'HE_khei'로 복사 또는 지정
    actual_col = khei_candidates[0]
    df['HE_khei'] = df[actual_col]
    print(f"✅ 성공! '{actual_col}' 변수를 'HE_khei'로 지정했습니다.")

    # 요약 통계량 확인
    print("\n--- 식생활평가지수(HE_khei) 요약 통계량 ---")
    print(df['HE_khei'].describe())
else:
    print("⚠️ 통합 파일 내에도 'khei' 관련 변수가 없습니다. 데이터 변수명을 다시 확인해 주세요.")

🔍 'khei'가 포함된 실제 변수 후보: []
⚠️ 통합 파일 내에도 'khei' 관련 변수가 없습니다. 데이터 변수명을 다시 확인해 주세요.


In [31]:
import glob

# 드라이브 전체에서 sas7bdat 파일 모두 찾기
all_sas_files = glob.glob('/content/drive/**/*.sas7bdat', recursive=True)
print("📂 드라이브 내 모든 SAS 파일 목록:")
for f in all_sas_files:
    print(f)

📂 드라이브 내 모든 SAS 파일 목록:
/content/drive/MyDrive/hdac_data/hn24_all.sas7bdat


In [32]:
# 통합 파일의 모든 변수 이름 불러오기
_, meta_all = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', metadataonly=True)
all_columns = meta_all.column_names

# 'he' 또는 'khei'가 포함된 변수 모두 찾기 (대소문자 무시)
matched_cols = [c for c in all_columns if 'he' in c.lower() or 'khei' in c.lower()]

print("🔍 'he' 또는 'khei'가 포함된 변수 목록:")
print(matched_cols)

🔍 'he' 또는 'khei'가 포함된 변수 목록:
['HE_fst', 'HE_mens', 'HE_prg', 'HE_dprg', 'HE_fh', 'HE_HPfh1', 'HE_HPfh2', 'HE_HPfh3', 'HE_HLfh1', 'HE_HLfh2', 'HE_HLfh3', 'HE_IHDfh1', 'HE_IHDfh2', 'HE_IHDfh3', 'HE_STRfh1', 'HE_STRfh2', 'HE_STRfh3', 'HE_DMfh1', 'HE_DMfh2', 'HE_DMfh3', 'HE_CNfh11', 'HE_CNfh12_1', 'HE_CNfh12_2', 'HE_CNfh21', 'HE_CNfh22_1', 'HE_CNfh22_2', 'HE_CNfh31', 'HE_CNfh32_1', 'HE_CNfh32_2', 'HE_CNfh32_3', 'HE_rPLS', 'HE_PLS_30', 'HE_mPLS', 'HE_nARM', 'HE_armcir', 'HE_sbp1', 'HE_dbp1', 'HE_sbp2', 'HE_dbp2', 'HE_sbp3', 'HE_dbp3', 'HE_sbp', 'HE_dbp', 'HE_HP', 'HE_ht', 'HE_wt', 'HE_wc', 'HE_BMI', 'HE_obe', 'HE_wt_pct', 'HE_BMI_pct', 'HE_glu', 'HE_HbA1c', 'HE_DM_HbA1c', 'HE_chol', 'HE_HDL_st2', 'HE_TG', 'HE_LDL_drct', 'HE_HCHOL', 'HE_HTG', 'HE_ast', 'HE_alt', 'HE_alt_etc', 'HE_hepaB', 'HE_hepaC', 'HE_HB', 'HE_HCT', 'HE_anem', 'HE_BUN', 'HE_crea', 'HE_WBC', 'HE_RBC', 'HE_Bplt', 'HE_MCV', 'HE_MCH', 'HE_MCHC', 'HE_PTH', 'HE_ALP', 'HE_Uacid', 'HE_hsCRP', 'HE_hsCRP_etc', 'HE_Ca', 'HE_P', 'HE_M

In [33]:
# 통합 데이터에서 식생활평가지수 총점 변수 'HEI'를 분석용 표준 이름 'HE_khei'로 매핑
if 'HEI' in df.columns:
    df['HE_khei'] = df['HEI']
    print("✅ 성공! 'HEI' 변수를 분석용 'HE_khei'로 지정했습니다.")

    # 요약 통계량 확인
    print("\n--- 식생활평가지수 총점(HEI) 요약 통계량 ---")
    print(df['HE_khei'].describe())
else:
    print("⚠️ 'HEI' 변수를 찾지 못했습니다.")

⚠️ 'HEI' 변수를 찾지 못했습니다.


In [34]:
# 1. 만약 전체 데이터프레임(df)에 'HEI'가 없거나 소문자 'hei'로 되어 있다면 찾아보기
hei_cols = [c for c in meta_all.column_names if c.lower() == 'hei']

if hei_cols:
    target_hei = hei_cols[0]
    print(f"🔍 발견된 식생활평가지수 변수명: {target_hei}")

    # 만약 현재 df에 이 변수가 없다면 다시 불러오거나 추가하기
    if target_hei not in df.columns:
        # 기존 df에 해당 컬럼만 추가로 붙여넣기
        df_temp, _ = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=[target_hei])
        df['HE_khei'] = df_temp[target_hei]
    else:
        df['HE_khei'] = df[target_hei]

    print("✅ 성공! 식생활평가지수(HE_khei) 매핑 완료.")

    # 요약 통계량 출력
    print("\n--- 식생활평가지수(HE_khei) 요약 통계량 ---")
    print(df['HE_khei'].describe())
else:
    print("⚠️ 파일 전체 메타데이터에서도 'HEI' 변수를 찾지 못했습니다.")

🔍 발견된 식생활평가지수 변수명: HEI
✅ 성공! 식생활평가지수(HE_khei) 매핑 완료.

--- 식생활평가지수(HE_khei) 요약 통계량 ---
count    5856.000000
mean       60.196119
std        13.028301
min        10.491610
25%        51.176407
50%        60.496841
75%        69.614304
max        97.490200
Name: HE_khei, dtype: float64
